### Inversion — gradiometry dispersion -> Vs(x, z)

Mirrors the das_ani pipeline (`inv_urban.ipynb` / `profiles_urban.ipynb`) so the
two methods are directly comparable. The gradiometry phase-velocity section from
`sloth.ipynb` is exported as per-position curves `results/inv_inputs/dispersion_*.txt`
(same format as das_ani's MASW picks); this notebook inverts them with the **same**
5-layer prior, CPSO settings, and `src.inv` plotting used for the urban data.

All heavy lifting lives in `src.inv` -- the notebook stays thin. Depth note: the
3-8 Hz band constrains only ~< 50 m, so we keep das_ani's 0-120 m axes for overlay
but flag everything below ~50 m as poorly resolved (`max_resolved_depth=50`).

In [ ]:
import glob, os, sys, pickle
import numpy as np
np.Inf = np.inf  # evodcinv shim on newer NumPy (matches das_ani)
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.interpolate import interp1d

from evodcinv import EarthModel, Layer, Curve
from disba import PhaseDispersion

sys.path.append('..')
from src.inv import (plot_predicted_curve, plot_model, plot_model_range,
                     model_param_range, get_mean_model, check,
                     save_profile_plot, plot_dispersion_fit, plot_2d_contour_section)

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'axes.labelsize': 14, 'axes.titlesize': 16,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12,
    'figure.dpi': 150, 'axes.linewidth': 1.2, 'pdf.fonttype': 42, 'ps.fonttype': 42,
})

#### 1. Prior model + single-station demo

The 5-layer prior is **identical to das_ani `inv_urban`** (thickness in km, Vs in
km/s, Poisson 0.32-0.34), so both methods invert into the same parameter space.
CPSO uses all cores (`workers=-1`) and `maxrun=5` for robustness.

In [ ]:
INPUT_DIR = "../results/inv_inputs"
file_list = sorted(glob.glob(os.path.join(INPUT_DIR, "dispersion_*.txt")))
print(f"{len(file_list)} dispersion curves in {INPUT_DIR}")

def build_model():
    """das_ani inv_urban prior: 5 layers, increasing velocity, fixed density."""
    m = EarthModel()
    m.add(Layer([0.005, 0.020], [0.10, 0.40], [0.32, 0.34]))
    m.add(Layer([0.010, 0.030], [0.20, 0.55], [0.32, 0.34]))
    m.add(Layer([0.015, 0.040], [0.30, 0.65], [0.32, 0.34]))
    m.add(Layer([0.015, 0.035], [0.40, 0.80], [0.32, 0.34]))
    m.add(Layer([0.050, 0.100], [0.50, 0.95], [0.32, 0.34]))
    m.configure(optimizer="cpso", misfit="rmse", density=lambda x: 1.0 + 0.0 * x,
                increasing_velocity=True,
                optimizer_args={"popsize": 20, "maxiter": 1500, "workers": -1, "seed": 42})
    return m

def load_curve(path):
    """Load a dispersion_*.txt -> (f_obs, v_obs_ms, evodcinv Curve) (period ascending)."""
    d = np.loadtxt(path, comments="#")
    f_obs, v_obs = d[:, 0], d[:, 1]
    curve = Curve(1.0 / f_obs[::-1], v_obs[::-1] / 1000.0, 0, "rayleigh")
    return f_obs, v_obs, curve

In [ ]:
# Single central station: invert and inspect convergence.
f_obs, v_obs, curve = load_curve(file_list[len(file_list) // 2])
res = build_model().invert([curve], maxrun=5)
print(res)
res.plot_misfit(); plt.show()

Dispersion fit (ensemble + best) and the 1-D Vs profile for this station.

In [ ]:
period = 1.0 / f_obs[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
plot_predicted_curve(res, period, 0, "rayleigh", "phase", show="percentage", percent=10,
                     plot_args={"xaxis": "frequency", "cmap": "turbo", "alpha": 0.1}, ax=ax)
plot_predicted_curve(res, period, 0, "rayleigh", "phase", show="best",
                     plot_args={"xaxis": "frequency", "color": "black", "linewidth": 2,
                                "linestyle": "--"}, ax=ax)
ax.plot(f_obs, v_obs, "ro", ms=6, label="Observed (gradiometry)")
h, l = ax.get_legend_handles_labels(); ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys())
ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Phase velocity (m/s)")
ax.set_title("Dispersion fit"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 6))
plot_model(res, "vs", show="best", zmax=60, plot_args={"color": "C3"}, ax=ax)
plot_model(res, "vs", show="mean", zmax=60, plot_args={"color": "C0"}, ax=ax)
ax.set_title("1-D Vs profile (single station)"); plt.tight_layout(); plt.show()

#### 2. Invert all positions -> 2-D Vs section

Same loop as das_ani `inv_urban`: per position, invert with `maxrun=5`, save the
1-D model, and map layers onto a 0-120 m grid by **layer tops** (the half-space
Vs fills below the deepest interface). Saves `final_2D_matrix.npz` + the result
list, exactly like the urban pipeline.

In [ ]:
OUT_DIR = "../results/inv_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
max_depth = 120
z_grid = np.arange(0, max_depth + 1, 1)

positions, vs_best, vs_mean, all_results = [], [], [], []
for fp in file_list:
    dist = float("".join(ch for ch in os.path.basename(fp) if ch.isdigit()))
    f_o, v_o, curve = load_curve(fp)
    res = build_model().invert([curve], maxrun=5)
    all_results.append(res); positions.append(dist)
    np.savetxt(os.path.join(OUT_DIR, f"1D_model_best_{dist:.0f}m.txt"), res.model,
               header="Thickness(km) Vp(km/s) Vs(km/s) Density(g/cm3)", fmt="%.6f")
    for model_arr, store in [(res.model, vs_best), (get_mean_model(res, 30), vs_mean)]:
        th = model_arr[:, 0] * 1000.0; vs = model_arr[:, 2] * 1000.0
        tops = np.insert(np.cumsum(th[:-1]), 0, 0)            # nodes at layer tops
        store.append(interp1d(tops, vs, kind="previous",
                              fill_value=(vs[0], vs[-1]), bounds_error=False)(z_grid))
    print(f"  x={dist:6.0f} m  misfit {res.misfit:.4f}")

positions = np.array(positions)
matrix_best = np.array(vs_best).T
matrix_mean = np.array(vs_mean).T
np.savez(os.path.join(OUT_DIR, "final_2D_matrix.npz"),
         vs_matrix_best=matrix_best, vs_matrix_mean=matrix_mean,
         positions=positions, z_grid=z_grid)
with open(os.path.join(OUT_DIR, "all_results_list.pkl"), "wb") as fh:
    pickle.dump(all_results, fh)
print(f"\ninverted {len(positions)} positions -> {OUT_DIR}/final_2D_matrix.npz")

#### 3. Check a station + dispersion fit (das_ani `profiles_urban` style)

In [ ]:
idx = len(positions) // 2
check(all_results, positions, idx)
plot_dispersion_fit(all_results, positions, idx, obs_dir=INPUT_DIR)

#### 4. 2-D Vs section

Rendered with the **same `plot_2d_contour_section`** as das_ani, so the gradiometry
and MASW sections can be compared on identical axes/colour scale. `max_resolved_depth=50`
shades the band-limited deep zone.

In [ ]:
plot_2d_contour_section(
    positions, z_grid, matrix_mean,
    max_depth=120, vmin=200, vmax=600,
    x_interp_step=2.0, smooth_sigma=(3.0, 15.0), smooth_units="physical",
    max_resolved_depth=50,            # 3-8 Hz -> ~< 50 m (cf. das_ani's 100 m)
    contour=True,
    save_path="../results/inv_profiles/contour_2D_profile_gradiometry.png",
)